# EEGPT Knowledge Distillation Pipeline

Multimodal knowledge distillation from EEG to fMRI using genomics-informed GNN.

- **Teacher**: Frozen EEGPT encoder (pretrained EEG transformer) with trainable linear head
- **Student**: ConnectivityGCN (EdgeConv-based GNN operating on fMRI brain graphs)
- **Evaluation**: Leave-One-Subject-Out (LOSO) cross-validation
- **Dataset**: NatView simultaneous EEG-fMRI, 22 subjects, binary vigilance (alert/drowsy)

## Setup

Upload the following to Kaggle Datasets before running:
1. `fmri_data/` - directory with `sub-*_interval_corr.npy` and `sub-*_labels.npy`
2. `gene_expression_schaefer210.npy` - gene expression matrix (210 ROIs x 294 genes)
3. `eegpt_mcae_58chs_4s_large4E.ckpt` - EEGPT pretrained checkpoint from Figshare

In [ ]:
# Install dependencies
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "torch-geometric", "torch-scatter", "torch-sparse"])

In [ ]:
# Clone EEGPT repository
import os

EEGPT_DIR = "/kaggle/working/EEGPT"
if not os.path.exists(EEGPT_DIR):
    os.system(f"git clone https://github.com/BINE022/EEGPT.git {EEGPT_DIR}")

# Add EEGPT to path
sys.path.insert(0, os.path.join(EEGPT_DIR, "downstream"))
sys.path.insert(0, EEGPT_DIR)

In [ ]:
import os
import sys
import glob
import tempfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Sequential, Linear, ReLU, BatchNorm1d
from torch_geometric.nn import EdgeConv, global_mean_pool
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from sklearn.metrics import (
    balanced_accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report,
)
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## 1. Configuration

Set paths and hyperparameters. Adjust `USE_SYNTHETIC` to `False` when real data is available.

In [ ]:
# Toggle synthetic vs real data
USE_SYNTHETIC = True

# Paths (Kaggle dataset layout)
FMRI_DIR = "/kaggle/input/fmri-data"  # sub-*_interval_corr.npy, sub-*_labels.npy
GENE_EXPR_PATH = "/kaggle/input/gene-expression/gene_expression_schaefer210.npy"
EEGPT_CKPT = "/kaggle/input/eegpt-checkpoint/eegpt_mcae_58chs_4s_large4E.ckpt"

# Hyperparameters
CONFIG = {
    "hidden_channels": 64,
    "num_classes": 2,
    "lr": 0.001,
    "weight_decay": 1e-4,
    "epochs": 30,
    "batch_size": 16,
    "temperature": 4.0,
    "alpha_phase1": 0.2,     # KD-heavy phase (epochs 0-19)
    "alpha_phase2": 0.8,     # CE-heavy phase (epochs 20-29)
    "phase1_epochs": 20,
    "feature_weight": 0.0,
    "k": 10,                 # top-k edges per node
    "num_nodes": 210,        # Schaefer 200 cortical + 10 subcortical
    "num_genes": 294,        # sleep/circadian genes
}

## 2. Student Model: ConnectivityGCN

3-layer EdgeConv GNN with BatchNorm, global mean pooling, adapted from teammate's architecture.
Returns both logits and intermediate features for knowledge distillation.

In [ ]:
class ConnectivityGCN(nn.Module):
    """EdgeConv-based GNN for brain graph classification."""

    def __init__(self, input_dim=1, hidden_channels=64, num_classes=2):
        super().__init__()
        self.hidden_channels = hidden_channels

        self.mlp1 = Sequential(Linear(2 * input_dim, hidden_channels), ReLU())
        self.mlp2 = Sequential(Linear(2 * hidden_channels, hidden_channels), ReLU())
        self.mlp3 = Sequential(Linear(2 * hidden_channels, hidden_channels), ReLU())

        self.conv1 = EdgeConv(self.mlp1, aggr='max')
        self.conv2 = EdgeConv(self.mlp2, aggr='max')
        self.conv3 = EdgeConv(self.mlp3, aggr='max')

        self.bn1 = BatchNorm1d(hidden_channels)
        self.bn2 = BatchNorm1d(hidden_channels)
        self.bn3 = BatchNorm1d(hidden_channels)

        self.classifier = Linear(hidden_channels, num_classes)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)

        x = self.conv3(x, edge_index)
        x = self.bn3(x)

        features = global_mean_pool(x, batch)
        logits = self.classifier(features)

        return {"logits": logits, "features": features}

## 3. Teacher Model: EEGPT Adapter

Frozen EEGPT encoder with a trainable linear classification head.
Produces 2048-dim features (512 embed_dim x 4 embed_num) and 2-class logits.

In [ ]:
EEGPT_CHANNELS = [
    'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC5', 'FC1', 'FC2',
    'FC6', 'T7', 'C3', 'Cz', 'C4', 'T8', 'TP9', 'CP5', 'CP1', 'CP2',
    'CP6', 'TP10', 'P7', 'P3', 'Pz', 'P4', 'P8', 'PO9', 'O1', 'Oz',
    'O2', 'PO10', 'AF7', 'AF3', 'AF4', 'AF8', 'F5', 'F1', 'F2', 'F6',
    'FT9', 'FT7', 'FC3', 'FC4', 'FT8', 'FT10', 'C5', 'C1', 'C2', 'C6',
    'TP7', 'CP3', 'CPz', 'CP4', 'TP8', 'P5', 'P1', 'P2',
]


def prepare_chan_ids(channel_names):
    """Map channel names to EEGPT indices."""
    name_to_idx = {name: i for i, name in enumerate(EEGPT_CHANNELS)}
    indices = [name_to_idx[n] for n in channel_names]
    return torch.tensor([indices], dtype=torch.long)


class EEGPTTeacher(nn.Module):
    """Frozen EEGPT encoder with trainable classification head."""

    def __init__(self, checkpoint_path=None, num_classes=2, embed_dim=512, embed_num=4):
        super().__init__()
        self.embed_dim = embed_dim
        self.embed_num = embed_num
        self.feature_dim = embed_dim * embed_num  # 2048
        self.num_classes = num_classes

        self.encoder = self._build_encoder(checkpoint_path)
        for param in self.encoder.parameters():
            param.requires_grad = False

        self.head = nn.Sequential(
            nn.LayerNorm(self.feature_dim),
            nn.Linear(self.feature_dim, num_classes),
        )

    def _build_encoder(self, checkpoint_path):
        from Modules.models.EEGPT_mcae import EEGTransformer
        encoder = EEGTransformer(
            img_size=[58, 1024], patch_size=64,
            embed_num=self.embed_num, embed_dim=self.embed_dim,
            depth=8, num_heads=8,
        )
        if checkpoint_path is not None:
            self._load_checkpoint(encoder, checkpoint_path)
        return encoder

    def _load_checkpoint(self, encoder, path):
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        state_dict = ckpt.get("state_dict", ckpt)
        prefix = "target_encoder."
        encoder_state = {
            k[len(prefix):]: v for k, v in state_dict.items() if k.startswith(prefix)
        }
        encoder.load_state_dict(encoder_state, strict=False)

    def extract_features(self, x, chan_ids=None):
        with torch.no_grad():
            if chan_ids is not None:
                chan_ids = chan_ids.to(x.device)
                if chan_ids.shape[0] == 1 and x.shape[0] > 1:
                    chan_ids = chan_ids.expand(x.shape[0], -1)
                out = self.encoder(x, chan_ids)
            else:
                out = self.encoder(x)
            pooled = out.mean(dim=1)
            return pooled.reshape(pooled.shape[0], -1)

    def forward(self, x, chan_ids=None):
        features = self.extract_features(x, chan_ids)
        logits = self.head(features)
        return {"logits": logits, "features": features}

    @torch.no_grad()
    def generate_teacher_cache(self, dataloader, chan_ids=None, device="cpu"):
        self.eval()
        self.to(device)
        all_logits, all_features, all_labels = [], [], []
        for batch in dataloader:
            if isinstance(batch, (list, tuple)):
                eeg, labels = batch[0], batch[1]
            else:
                eeg, labels = batch, None
            eeg = eeg.to(device)
            out = self.forward(eeg, chan_ids)
            all_logits.append(out["logits"].cpu())
            all_features.append(out["features"].cpu())
            if labels is not None:
                all_labels.append(labels.cpu())
        result = {"logits": torch.cat(all_logits), "features": torch.cat(all_features)}
        if all_labels:
            result["labels"] = torch.cat(all_labels)
        return result

## 4. Graph Dataset

Loads fMRI correlation matrices as PyG graphs with top-k edges.
Optionally concatenates gene expression as additional node features.

In [ ]:
class VigilanceGraphDataset(Dataset):
    """PyG dataset of brain graphs for vigilance classification."""

    def __init__(self, fmri_path, gene_expression_path=None, k=10):
        super().__init__()
        self.k = k
        self.samples = []
        self._subjects = []

        self.gene_expression = None
        if gene_expression_path is not None and os.path.exists(gene_expression_path):
            self.gene_expression = torch.FloatTensor(np.load(gene_expression_path))

        corr_files = sorted(glob.glob(os.path.join(fmri_path, "sub-*_interval_corr.npy")))
        for corr_file in corr_files:
            filename = os.path.basename(corr_file)
            subject_id = filename.split("_")[0]
            corr_intervals = np.load(corr_file)

            label_file = os.path.join(fmri_path, f"{subject_id}_labels.npy")
            if not os.path.exists(label_file):
                continue
            labels = np.load(label_file)

            for i in range(len(corr_intervals)):
                if i >= len(labels):
                    break
                self.samples.append({
                    "subject": subject_id, "interval": i,
                    "corr_matrix": corr_intervals[i], "label": int(labels[i]),
                })
                if subject_id not in self._subjects:
                    self._subjects.append(subject_id)

    def len(self):
        return len(self.samples)

    def get(self, idx):
        sample = self.samples[idx]
        corr = sample["corr_matrix"]
        edge_index, edge_attr = self._corr_to_graph(corr, self.k)
        mean_conn = torch.FloatTensor(corr.mean(axis=1)).unsqueeze(1)

        if self.gene_expression is not None:
            gene_feat = self.gene_expression[:mean_conn.shape[0]]
            x = torch.cat([mean_conn, gene_feat], dim=1)
        else:
            x = mean_conn

        y = torch.tensor(sample["label"], dtype=torch.long)
        data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
        data.subject = sample["subject"]
        data.interval = sample["interval"]
        return data

    def _corr_to_graph(self, corr_matrix, k):
        num_nodes = corr_matrix.shape[0]
        edges, weights = [], []
        for node in range(num_nodes):
            vals = np.abs(corr_matrix[node, :].copy())
            vals[node] = 0.0
            top_k = np.argsort(vals)[-k:]
            for nb in top_k:
                edges.append((node, nb))
                weights.append(corr_matrix[node, nb])
        return (
            torch.tensor(np.array(edges).T, dtype=torch.long),
            torch.tensor(weights, dtype=torch.float32),
        )

    def get_subjects(self):
        return list(self._subjects)

    def get_subject_indices(self, subject_id):
        return [i for i, s in enumerate(self.samples) if s["subject"] == subject_id]

    @property
    def num_node_features(self):
        if not self.samples:
            return 1
        return self.get(0).x.shape[1]

In [ ]:
def create_synthetic_dataset(num_subjects=5, intervals_per_subject=10,
                              num_nodes=210, num_genes=0, k=10):
    """Generate synthetic data for testing the pipeline."""
    save_dir = tempfile.mkdtemp(prefix="synthetic_brain_")

    gene_path = None
    if num_genes > 0:
        gene_expr = np.random.randn(num_nodes, num_genes).astype(np.float32)
        gene_path = os.path.join(save_dir, "gene_expression.npy")
        np.save(gene_path, gene_expr)

    for subj_idx in range(num_subjects):
        sid = f"sub-{subj_idx + 1:02d}"
        corr_list = []
        for _ in range(intervals_per_subject):
            r = np.random.randn(num_nodes, num_nodes).astype(np.float32)
            c = (r + r.T) / 2
            np.fill_diagonal(c, 1.0)
            corr_list.append(np.clip(c, -1, 1))
        np.save(os.path.join(save_dir, f"{sid}_interval_corr.npy"), np.stack(corr_list))
        np.save(os.path.join(save_dir, f"{sid}_labels.npy"),
                np.random.randint(0, 2, size=intervals_per_subject))

    return VigilanceGraphDataset(save_dir, gene_expression_path=gene_path, k=k)

## 5. KD Loss and Metrics

In [ ]:
class KDLoss(nn.Module):
    """Knowledge distillation loss: alpha*CE + (1-alpha)*KL*T^2."""

    def __init__(self, alpha=0.5, temperature=4.0, feature_weight=0.0):
        super().__init__()
        self.alpha = alpha
        self.temperature = temperature
        self.feature_weight = feature_weight
        self.ce_loss = nn.CrossEntropyLoss()
        self.kl_loss = nn.KLDivLoss(reduction="batchmean")
        self._projector = None

    def forward(self, student_logits, teacher_logits, true_labels,
                student_feats=None, teacher_feats=None):
        hard_loss = self.ce_loss(student_logits, true_labels)

        T = self.temperature
        s_soft = F.log_softmax(student_logits / T, dim=1)
        t_soft = F.softmax(teacher_logits / T, dim=1)
        soft_loss = self.kl_loss(s_soft, t_soft) * (T * T)

        total = self.alpha * hard_loss + (1.0 - self.alpha) * soft_loss

        if self.feature_weight > 0 and student_feats is not None and teacher_feats is not None:
            if student_feats.shape[1] != teacher_feats.shape[1]:
                if self._projector is None or self._projector.in_features != student_feats.shape[1]:
                    self._projector = nn.Linear(
                        student_feats.shape[1], teacher_feats.shape[1]
                    ).to(student_feats.device)
                proj = self._projector(student_feats)
            else:
                proj = student_feats
            total = total + self.feature_weight * F.mse_loss(proj, teacher_feats.detach())

        return total


def compute_metrics(y_true, y_pred, y_prob=None):
    """Compute balanced accuracy, F1, and AUROC."""
    result = {
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }
    if y_prob is not None:
        try:
            result["auroc"] = float(roc_auc_score(y_true, y_prob[:, 1]))
        except ValueError:
            result["auroc"] = float("nan")
    return result

## 6. LOSO Trainer

Leave-One-Subject-Out cross-validation with two-phase alpha schedule:
- Phase 1 (epochs 0-19): KD-heavy, alpha=0.2 (80% soft label weight)
- Phase 2 (epochs 20-29): CE-heavy, alpha=0.8 (80% hard label weight)

In [ ]:
def train_loso(dataset, student_class, config, teacher_cache=None):
    """Run LOSO cross-validation.

    Args:
        dataset: VigilanceGraphDataset
        student_class: GNN class
        config: hyperparameter dict
        teacher_cache: dict with 'logits'/'features' or None for baseline

    Returns:
        dict with fold_results, mean/std balanced accuracy, mode
    """
    subjects = dataset.get_subjects()
    fold_results = []
    mode = "kd" if teacher_cache is not None else "baseline"
    device = config["device"] if "device" in config else DEVICE

    print(f"\nRunning LOSO CV ({mode} mode) across {len(subjects)} subjects")
    print("=" * 60)

    for fold_idx, test_subj in enumerate(subjects):
        test_idx = dataset.get_subject_indices(test_subj)
        train_idx = [i for i in range(len(dataset)) if i not in set(test_idx)]

        if not test_idx or not train_idx:
            continue

        train_data = [dataset[i] for i in train_idx]
        test_data = [dataset[i] for i in test_idx]

        # Attach teacher outputs to Data objects (survives shuffling)
        kd_mode = teacher_cache is not None
        if kd_mode:
            for i, gi in enumerate(train_idx):
                train_data[i].teacher_logits = teacher_cache["logits"][gi]
                train_data[i].teacher_features = teacher_cache["features"][gi]

        train_loader = DataLoader(train_data, batch_size=config["batch_size"], shuffle=True)
        test_loader = DataLoader(test_data, batch_size=config["batch_size"], shuffle=False)

        # Build fresh student
        input_dim = train_data[0].x.shape[1]
        student = student_class(
            input_dim=input_dim,
            hidden_channels=config["hidden_channels"],
            num_classes=config["num_classes"],
        ).to(device)
        optimizer = torch.optim.Adam(
            student.parameters(), lr=config["lr"], weight_decay=config["weight_decay"]
        )
        ce_criterion = nn.CrossEntropyLoss()

        # Train
        for epoch in range(config["epochs"]):
            student.train()
            if kd_mode:
                alpha = config["alpha_phase1"] if epoch < config["phase1_epochs"] else config["alpha_phase2"]
            else:
                alpha = 1.0

            kd_crit = KDLoss(
                alpha=alpha, temperature=config["temperature"],
                feature_weight=config["feature_weight"],
            ).to(device)

            for batch in train_loader:
                batch = batch.to(device)
                optimizer.zero_grad()
                out = student(batch)

                if kd_mode:
                    loss = kd_crit(
                        out["logits"], batch.teacher_logits, batch.y,
                        student_feats=out["features"],
                        teacher_feats=batch.teacher_features,
                    )
                else:
                    loss = ce_criterion(out["logits"], batch.y)

                loss.backward()
                optimizer.step()

        # Evaluate
        student.eval()
        all_preds, all_labels, all_probs = [], [], []
        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(device)
                out = student(batch)
                probs = torch.softmax(out["logits"], dim=1)
                all_preds.append(out["logits"].argmax(dim=1).cpu())
                all_labels.append(batch.y.cpu())
                all_probs.append(probs.cpu())

        y_true = torch.cat(all_labels).numpy()
        y_pred = torch.cat(all_preds).numpy()
        y_prob = torch.cat(all_probs).numpy()

        metrics = compute_metrics(y_true, y_pred, y_prob)
        metrics["subject"] = test_subj
        metrics["fold"] = fold_idx
        fold_results.append(metrics)

        print(
            f"  Fold {fold_idx + 1}/{len(subjects)} [{test_subj}]: "
            f"bal_acc={metrics['balanced_accuracy']:.3f}, f1={metrics['f1']:.3f}"
        )

    bal_accs = [r["balanced_accuracy"] for r in fold_results]
    mean_ba = np.mean(bal_accs) if bal_accs else 0.0
    std_ba = np.std(bal_accs) if bal_accs else 0.0

    print("=" * 60)
    print(f"Mean balanced accuracy: {mean_ba:.3f} +/- {std_ba:.3f}")

    return {
        "fold_results": fold_results,
        "mean_balanced_accuracy": float(mean_ba),
        "std_balanced_accuracy": float(std_ba),
        "mode": mode,
    }

## 7. Load Dataset

In [ ]:
if USE_SYNTHETIC:
    print("Using synthetic data for pipeline testing")
    dataset = create_synthetic_dataset(
        num_subjects=5, intervals_per_subject=10,
        num_nodes=CONFIG["num_nodes"], num_genes=CONFIG["num_genes"],
        k=CONFIG["k"],
    )
else:
    gene_path = GENE_EXPR_PATH if os.path.exists(GENE_EXPR_PATH) else None
    dataset = VigilanceGraphDataset(FMRI_DIR, gene_expression_path=gene_path, k=CONFIG["k"])

print(f"Dataset: {len(dataset)} samples, {len(dataset.get_subjects())} subjects")
print(f"Node features: {dataset.get(0).x.shape[1]}")
print(f"Subjects: {dataset.get_subjects()}")

# Set input_dim from data
CONFIG["input_dim"] = dataset.get(0).x.shape[1]
CONFIG["device"] = DEVICE

## 8. Generate Teacher Cache (or Synthetic)

When real EEG data and EEGPT checkpoint are available, use the teacher model to generate
soft labels. For synthetic testing, we generate random teacher outputs.

In [ ]:
if USE_SYNTHETIC:
    # Generate random teacher outputs for testing
    n_samples = len(dataset)
    teacher_cache = {
        "logits": torch.randn(n_samples, CONFIG["num_classes"]),
        "features": torch.randn(n_samples, 2048),
    }
    print(f"Generated synthetic teacher cache: {n_samples} samples")
else:
    # Load real EEGPT teacher and generate cache
    # NOTE: This requires EEG data loaded into a DataLoader
    # and the EEGPT checkpoint downloaded
    teacher = EEGPTTeacher(checkpoint_path=EEGPT_CKPT, num_classes=2).to(DEVICE)
    # TODO: Load EEG data and create eeg_dataloader
    # chan_ids = prepare_chan_ids(EEGPT_CHANNELS)
    # teacher_cache = teacher.generate_teacher_cache(eeg_dataloader, chan_ids, DEVICE)
    raise NotImplementedError("Wire up EEG DataLoader for real teacher cache generation")

## 9. Run KD Training (LOSO)

In [ ]:
kd_results = train_loso(dataset, ConnectivityGCN, CONFIG, teacher_cache=teacher_cache)

## 10. Run Baseline (No KD)

In [ ]:
baseline_results = train_loso(dataset, ConnectivityGCN, CONFIG, teacher_cache=None)

## 11. Results Comparison

In [ ]:
def print_comparison(kd_res, baseline_res):
    """Print side-by-side comparison of KD vs baseline results."""
    print("\n" + "=" * 60)
    print("RESULTS COMPARISON: KD vs Baseline")
    print("=" * 60)

    print(f"\n{'Metric':<25} {'KD':>12} {'Baseline':>12} {'Delta':>12}")
    print("-" * 61)

    for metric in ["mean_balanced_accuracy", "std_balanced_accuracy"]:
        kd_val = kd_res[metric]
        bl_val = baseline_res[metric]
        delta = kd_val - bl_val
        label = metric.replace("_", " ").title()
        print(f"{label:<25} {kd_val:>12.4f} {bl_val:>12.4f} {delta:>+12.4f}")

    # Per-fold comparison
    print(f"\n{'Fold':<6} {'Subject':<10} {'KD Acc':>10} {'BL Acc':>10} {'Delta':>10}")
    print("-" * 46)
    for kd_f, bl_f in zip(kd_res["fold_results"], baseline_res["fold_results"]):
        d = kd_f["balanced_accuracy"] - bl_f["balanced_accuracy"]
        print(
            f"{kd_f['fold']+1:<6} {kd_f['subject']:<10} "
            f"{kd_f['balanced_accuracy']:>10.3f} {bl_f['balanced_accuracy']:>10.3f} "
            f"{d:>+10.3f}"
        )

print_comparison(kd_results, baseline_results)

## 12. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fold comparison bar chart
subjects = [r["subject"] for r in kd_results["fold_results"]]
kd_accs = [r["balanced_accuracy"] for r in kd_results["fold_results"]]
bl_accs = [r["balanced_accuracy"] for r in baseline_results["fold_results"]]

x = np.arange(len(subjects))
width = 0.35
axes[0].bar(x - width/2, kd_accs, width, label="KD", color="steelblue")
axes[0].bar(x + width/2, bl_accs, width, label="Baseline", color="coral")
axes[0].set_xlabel("Subject")
axes[0].set_ylabel("Balanced Accuracy")
axes[0].set_title("Per-Fold Balanced Accuracy")
axes[0].set_xticks(x)
axes[0].set_xticklabels(subjects, rotation=45, ha="right")
axes[0].legend()
axes[0].set_ylim(0, 1)

# Aggregate comparison
means = [kd_results["mean_balanced_accuracy"], baseline_results["mean_balanced_accuracy"]]
stds = [kd_results["std_balanced_accuracy"], baseline_results["std_balanced_accuracy"]]
bars = axes[1].bar(["KD", "Baseline"], means, yerr=stds, capsize=8,
                    color=["steelblue", "coral"], edgecolor="black")
axes[1].set_ylabel("Mean Balanced Accuracy")
axes[1].set_title("Aggregate: KD vs Baseline")
axes[1].set_ylim(0, 1)
for bar, m in zip(bars, means):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f"{m:.3f}", ha="center", fontsize=11)

plt.tight_layout()
plt.savefig("kd_vs_baseline.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figure: kd_vs_baseline.png")

In [ ]:
# Aggregate confusion matrices across all folds
# (collecting all predictions for a single summary matrix)
def plot_aggregate_confusion(results, title="Confusion Matrix"):
    """Plot aggregate confusion matrix from all LOSO folds."""
    # Re-run to collect predictions (not stored in fold_results)
    # For now, show the per-fold metric summary
    print(f"\n{title}")
    print(f"{'Fold':<6} {'Subject':<10} {'Bal.Acc':<10} {'F1':<10} {'AUROC':<10}")
    print("-" * 46)
    for r in results["fold_results"]:
        auroc = f"{r.get('auroc', float('nan')):.3f}"
        print(
            f"{r['fold']+1:<6} {r['subject']:<10} "
            f"{r['balanced_accuracy']:.3f}     {r['f1']:.3f}     {auroc}"
        )

plot_aggregate_confusion(kd_results, "KD Model - Per-Fold Metrics")
plot_aggregate_confusion(baseline_results, "Baseline Model - Per-Fold Metrics")

## Summary

This notebook implements the full EEGPT Knowledge Distillation pipeline:

1. **Teacher**: Frozen EEGPT encoder extracts 2048-dim features from EEG, classified via linear head
2. **Student**: ConnectivityGCN (3-layer EdgeConv) operates on fMRI brain graphs with optional gene expression node features
3. **KD Loss**: alpha * CE(hard labels) + (1-alpha) * KL(soft labels) * T^2
4. **Two-phase schedule**: KD-heavy (alpha=0.2) for first 20 epochs, CE-heavy (alpha=0.8) for last 10
5. **LOSO CV**: One subject held out per fold across all 22 NatView subjects

### Next steps:
- Set `USE_SYNTHETIC = False` and provide real fMRI data paths
- Wire up EEG DataLoader for teacher cache generation
- Run ablation: connectivity-only vs connectivity+genomics node features
- Tune hyperparameters (hidden_channels, temperature, alpha schedule)